# Imports

In [0]:
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import os
import plotly.io as pio
from pathlib import Path
import pyspark.sql.functions as F

# Data Import

In [0]:
bank_data = spark.table('workspace.default.fandc_raw_data').select(
    F.col('Date'),
    F.col('Type').cast('string'),
    F.col('Description').cast('string'),
    F.col('Category').cast('string'),
    F.col('Amount').try_cast('double'),
    F.col('Note').cast('string'),
    F.col('_rescued_data')
).withColumn('Date', F.to_date(F.col('Date'), 'MM-dd-yyyy')).withColumn(
    'Description', F.regexp_replace(F.col('Description'), '"', '')
).dropDuplicates().toPandas()

In [0]:
chase_data = spark.table('workspace.default.chase_raw_data').select(
    F.col('Transaction_Date').cast('string'),
    F.col('Post_Date').cast('string'),
    F.col('Description').cast('string'),
    F.col('Category').cast('string'),
    F.col('Type').cast('string'),
    F.col('Amount').cast('double')
).withColumn(
    'Transaction_Date', F.to_date(F.col('Transaction_Date'), 'MM/dd/yyyy')
).withColumn(
    'Post_Date', F.to_date(F.col('Post_Date'), 'MM/dd/yyyy')
).withColumn(
    'Description', F.regexp_replace(F.col('Description'), '"', '')
).dropDuplicates()

In [0]:
tos_data = spark.table('workspace.default.tos_raw_data').select(
    F.col('Transaction_Date'),
    F.col('Action').cast('string'),
    F.col('Symbol').cast('string'),
    F.col('Description').cast('string'),
    F.col('Quantity').cast('double'),
    F.col('Price').cast('double')
)

In [0]:
yahoo_data_spark = spark.table('workspace.default.yahoo_raw_data').select(
    F.col('Symbol').cast('string'),
    F.col('Current_Price').cast('double'),
    F.col('Date').cast('string'),
    F.col('Time').cast('string'),
    F.col('Change').cast('string'),
    F.col('Open').cast('double'),
    F.col('High').cast('double'),
    F.col('Low').cast('double'),
    F.col('Volume').cast('double'),
    F.col('Trade_Date').cast('string'),
    F.col('Purchase_Price').cast('double'),
    F.col('Quantity').cast('double'),
    F.col('High_Limit').cast('double'),
    F.col('Low_Limit').cast('double'),
    F.col('Comment').cast('string')
).withColumn(
    'Date', F.to_date(F.col('Date'), 'yyyy/MM/dd')
).withColumn(
    'Trade_Date', F.to_date(F.col('Trade_Date'), 'yyyyMMdd')
).dropDuplicates()

In [0]:
vanguard_data_spark = spark.table('workspace.default.vanguard_raw_data').select(
    F.col('Account_Number').cast('string'),
    F.col('Trade_Date').cast('string'),
    F.col('Settlement_Date').cast('string'),
    F.col('Transaction_Type').cast('string'),
    F.col('Transaction_Description').cast('string'),
    F.col('Investment_Name').cast('string'),
    F.col('Symbol').cast('string'),
    F.col('Shares').cast('double'),
    F.col('Share_Price').cast('double'),
    F.col('Principal_Amount').cast('double'),
    F.col('Commissions_and_Fees').cast('double'),
    F.col('Net_Amount').cast('double'),
    F.col('Accrued_Interest').cast('double'),
    F.col('Account_Type').cast('string')
).withColumn(
    'Date', F.to_date(F.col('Settlement_Date'), 'yyyy-MM-dd')
).withColumn(
    'Trade_Date', F.to_date(F.col('Trade_Date'), 'yyyy-MM-dd')
).dropDuplicates()

# Analysis

## Bank Data

In [0]:
totals_desc = bank_data.groupby('Description').apply(lambda x: x['Amount'].sum()).reset_index()

In [0]:
money_made = bank_data.loc[bank_data['Amount']>0]
#money_made['datetime']=money_made.apply(lambda x: pd.to_datetime(x['Date'], format = "%m-%d-%Y"), axis=1)
money_made_grouped = money_made.groupby('Type').agg(
    total = pd.NamedAgg('Amount', lambda x: x.sum()),
    min_date = pd.NamedAgg('Date', lambda x: pd.to_datetime(x).min()),
    max_date = pd.NamedAgg('Date', lambda x: pd.to_datetime(x).max())
).reset_index()

In [0]:
money_made_grouped['month_diff']=money_made_grouped.apply(lambda x: ((x['max_date']-x['min_date']) / np.timedelta64(1, 'D')) / 30.4, axis=1)
money_made_grouped['monthly_make']=money_made_grouped.apply(lambda x: x['total']/round(x['month_diff']), axis=1)

money_spent= bank_data.loc[(bank_data['Amount']<0)&(bank_data['Description']!='VANGUARD BUY INVESTMENT')&(bank_data['Description']!="SCHWAB BROKERAGE MONEYLINK")].drop_duplicates()
money_spent['datetime']=money_spent.apply(lambda x: pd.to_datetime(x['Date'], format = "%Y-%m-%d"), axis=1)

money_spent_no_cc = money_spent.loc[money_spent['Description']!='CHASE CREDIT CRD EPAY'].drop_duplicates()

In [0]:
display(money_spent)

In [0]:
tt = money_spent.groupby(money_spent.datetime.dt.year).agg(
    total_spent = pd.NamedAgg('Amount', lambda x: abs(sum(x)))
).reset_index()

In [0]:
display(tt)

In [0]:
money_spent['year'] = money_spent['datetime'].dt.year
money_spent['month'] = money_spent['datetime'].dt.month

money_spent_monthly = money_spent.groupby(['year', 'month']).agg(
    total_spent=('Amount', lambda x: abs(sum(x)))
).reset_index()

display(money_spent_monthly)

## Vanguard Data

In [0]:
vanguard_data = vanguard_data_spark.toPandas()

In [0]:
van_div = vanguard_data.loc[(vanguard_data['Transaction_Description']=='Dividend Received') & (vanguard_data['Symbol']!='VTSAX')]
van_div['datetime']=van_div.apply(lambda x: pd.to_datetime(x['Date'], format = "%Y-%m-%d"), axis=1)

In [0]:
van_div['year'] = van_div['datetime'].dt.year
van_div['month'] = van_div['datetime'].dt.month

van_div_monthly = van_div.groupby(
    ['year', 'month']
).agg(
    total_received=('Net_Amount', lambda x: abs(sum(x)))
).reset_index()

van_div_yearly = van_div.groupby(
    ['year']
).agg(
    total_received=('Net_Amount', lambda x: abs(sum(x)))
).reset_index()

In [0]:
display(van_div_yearly)

In [0]:
display(van_div_monthly)

In [0]:
display(vanguard_data)

## Chase Date

In [0]:
display(chase_data)

In [0]:
chase_sales = chase_data.where(chase_data['Type']=='Sale').toPandas()
chase_sales['datetime']=chase_sales.apply(lambda x: pd.to_datetime(x['Transaction_Date'], format = "%Y-%m-%d"), axis=1)
chase_sales['year'] = chase_sales['datetime'].dt.year
chase_sales['month'] = chase_sales['datetime'].dt.month

chase_sales_monthly = chase_sales.groupby(
    ['year', 'month']
).agg(
    total_spent=('Amount', lambda x: abs(sum(x)))
).reset_index()

chase_sales_yearly = chase_sales.groupby(
    ['year']
).agg(
    total_spent=('Amount', lambda x: abs(sum(x)))
).reset_index()

In [0]:
display(chase_sales_yearly)

## Yahoo Data Calculations

In [0]:
def calc_total(x , y):
    return float(x) * float(y)
yahoo_data = yahoo_data_spark.toPandas()
yahoo_data['Current_Price'].fillna(1,inplace=True)
yahoo_data['purchase_total']=yahoo_data.apply(lambda x: x['Purchase_Price']*x['Quantity'], axis=1)
yahoo_data['Trade_Date']=yahoo_data.loc[yahoo_data['Trade_Date'].notna()].apply(lambda x: dt.datetime.strptime(str(x['Trade_Date']), '%Y-%m-%d'), axis=1)
comp_data = yahoo_data.groupby('Symbol').agg(
    total_amount_shares = pd.NamedAgg('Quantity', aggfunc = lambda x: sum(x)),
    current_price = pd.NamedAgg('Current_Price', aggfunc = lambda x: max(x)),
    total_spent = pd.NamedAgg('purchase_total',aggfunc = lambda x: sum(x)),
).reset_index().fillna(1)

comp_data_2 = comp_data.groupby(['Symbol']).apply(lambda x: pd.Series(dict(
    total_amount_per_share = calc_total(x['total_amount_shares'],x['current_price'])
))).reset_index()
#.T.astype('int')
comp_final = comp_data.merge(comp_data_2, how='inner',on=['Symbol'])
comp_final['total_gained']=comp_final.apply(lambda x: x['total_amount_per_share']-x['total_spent'], axis=1)
yahoo_data['Total_Spent'] = yahoo_data.apply(lambda x: np.multiply(x['Quantity'],x['Purchase_Price']), axis=1)
yahoo_data['total_current_value'] = yahoo_data.apply(lambda x: np.multiply(x['Quantity'],x['Current_Price']), axis=1)
yahoo_data_linear = yahoo_data[['Symbol','Date','Trade_Date','Purchase_Price','Quantity','Total_Spent','total_current_value']].copy()
yahoo_data_linear['Trade_Date'].fillna(dt.date(2000,1,1),inplace=True)
yahoo_data_linear['year']=yahoo_data_linear['Trade_Date'].apply(lambda x: str(x)[0:4])
total_yearly_spent = yahoo_data_linear.groupby('year').agg(
    total_spent_yearly = pd.NamedAgg('Total_Spent', aggfunc = lambda x: np.sum(x))
).reset_index()

In [0]:
yahoo_data['total_current_value'].cumsum()

In [0]:
display(yahoo_data)

In [0]:
# Load data
end_data = dt.datetime.now()
history = yahoo_data_linear.copy()
history["Date"] = pd.to_datetime(history["Trade_Date"], format="%Y-%m-%d")
history=history.loc[history['Date']>"1999-10-01"]
history.sort_values(by=["Date"], ascending=True,inplace=True)
history['cumsum']=history['Total_Spent'].cumsum()
history['total_value']=history['total_current_value'].cumsum()
# Filter out voo purchases
voo_purchases = history.loc[(history["Symbol"] == "VOO") & (history['Total_Spent']>5000)].copy()
voo_purchases_2=voo_purchases[['Symbol','Date','Total_Spent']].copy()
voo_mi = pd.MultiIndex.from_frame(voo_purchases_2[['Symbol','Date','Total_Spent']])

history.set_index("Date",inplace=True)
# Plot Dow Jones average against time
fig = go.Figure()
fig.add_trace(go.Scatter(x=history.index, y=history["cumsum"], mode="lines", name="invested_total"))
fig.add_trace(go.Scatter(x=history.index, y=history["total_value"], mode="lines", name="invested_total_value"))
#fig.add_trace(go.Scatter(x=yearly_income_plot.index, y=yearly_income_plot['cumsum'], mode= "lines",name="income_earned"))
"""fig.add_hline(y=1, line_dash="dot", row=3, col="all",
              annotation_text="Jan 1, 2018 baseline", 
              annotation_position="bottom right")"""
# Add vertical bars for recession months
for month in voo_mi:
    end_day = month[1] + dt.timedelta(days=1)
    fig.add_vrect(x0=month[1], x1 = end_day ,row="all", col=1,
              fillcolor="black", opacity=0.75, line_width=1, annotation_text = month[2], annotation_textangle = 90)

# Update layout
fig.update_layout(
xaxis=dict(title="Date"),
yaxis=dict(title="Total Spent"),
title="Total Invested Over Time",
)

# Show plot
fig.show()